In [1]:
from huggingface_hub import snapshot_download
import os

model_id = "meta-llama/Llama-3.2-1B-Instruct"

local_path = snapshot_download(repo_id=model_id)

print("downloaded to:", local_path)
print("files on disk:")
for f in os.listdir(local_path):
    size_mb = os.path.getsize(os.path.join(local_path, f)) / (1024 * 1024)
    print(f"  {f:40s} {size_mb:>10.2f} MB")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

downloaded to: /home/ubuntu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6
files on disk:
  USE_POLICY.md                                  0.01 MB
  special_tokens_map.json                        0.00 MB
  generation_config.json                         0.00 MB
  tokenizer.json                                 8.66 MB
  tokenizer_config.json                          0.05 MB
  original                                       0.00 MB
  model.safetensors                           2357.14 MB
  LICENSE.txt                                    0.01 MB
  README.md                                      0.04 MB
  .gitattributes                                 0.00 MB
  config.json                                    0.00 MB


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, psutil, os

model_id = "meta-llama/Llama-3.2-1B-Instruct"

proc = psutil.Process(os.getpid())
ram_before = proc.memory_info().rss / (1024**3)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)
model.eval()

ram_after = proc.memory_info().rss / (1024**3)

print(f"RAM before load: {ram_before:.2f} GB")
print(f"RAM after  load: {ram_after:.2f} GB")
print(f"model occupies : {ram_after - ram_before:.2f} GB")
print("model dtype    :", next(model.parameters()).dtype)
print("model device   :", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

RAM before load: 0.68 GB
RAM after  load: 3.17 GB
model occupies : 2.48 GB
model dtype    : torch.float16
model device   : cpu


In [4]:
text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print("========== INPUT ==========")
print("text  :", repr(text))
print("shape :", input_ids.shape)
print("dtype :", input_ids.dtype)
print("ndim  :", input_ids.ndim)
print("===========================")

print()

print("========== OUTPUT ==========")
print("text  :", repr(tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)))
print("shape :", output_ids.shape)
print("dtype :", output_ids.dtype)
print("ndim  :", output_ids.ndim)
print("============================")

========== INPUT ==========
text  : 'The capital of France is'
shape : torch.Size([1, 6])
dtype : torch.int64
ndim  : 2

========== OUTPUT ==========
text  : 'The capital of France is Paris'
shape : torch.Size([1, 7])
dtype : torch.int64
ndim  : 2


In [5]:
text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

# ---------- Step 1: LLM (opaque) — outputs a token ID ----------
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_token_id = output_ids[0, -1].item()

# ---------- Step 2: decode — inverse vocab lookup ----------
text_piece = tokenizer.decode([new_token_id], clean_up_tokenization_spaces=False)


print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                    STEP 1: LLM (opaque)                  @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

print("========== INPUT ==========")
print("text  :", repr(text))
print("===========================")

print()

print("========== OUTPUT ==========")
print("token ID :", new_token_id)
print("type     :", type(new_token_id).__name__)
print("============================")

print()
print()

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                       STEP 2: decode                     @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

print("========== INPUT ==========")
print("token ID :", new_token_id)
print("type     :", type(new_token_id).__name__)
print("===========================")

print()

print("========== OUTPUT ==========")
print("text piece :", repr(text_piece))
print("============================")

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
@                    STEP 1: LLM (opaque)                  @
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
========== INPUT ==========
text  : 'The capital of France is'

========== OUTPUT ==========
token ID : 12366
type     : int


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
@                       STEP 2: decode                     @
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
========== INPUT ==========
token ID : 12366
type     : int

========== OUTPUT ==========
text piece : ' Paris'
